In [48]:
import pandas as pd
import numpy as np
import joblib
import os

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier
)
from sklearn.tree import DecisionTreeClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

print("Libraries imported successfully")


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.5/101.7 MB 419.0 kB/s eta 0:04:02
   ---------------------------------------- 0.5/101.7 MB 419.0 kB/s eta 0:04:02
   ---------------------------------------- 0.8/101.7 MB 405.1 kB/s eta 0:04:10
   ---------------------------------------- 0.8/101.7 MB 405.1 kB/s eta 0:04:10
   ---------------------------------------- 0.8/101.7 MB 405.1 kB/s eta 0:04:10
   ---------------------------------------- 1.0/101.7 MB 430.8 kB/s eta 0:03:54
   --------------------

In [49]:
df = pd.read_csv("D:\hospital_dashboard\dataset\diabetic_data.csv")

print("Dataset shape:", df.shape)

df.head()

<>:1: SyntaxWarning: "\h" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\h"? A raw string is also an option.
<>:1: SyntaxWarning: "\h" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\h"? A raw string is also an option.
C:\Users\vishnuvardhan\AppData\Local\Temp\ipykernel_43848\780056841.py:1: SyntaxWarning: "\h" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\h"? A raw string is also an option.
  df = pd.read_csv("D:\hospital_dashboard\dataset\diabetic_data.csv")


Dataset shape: (101766, 50)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [50]:
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum().sort_values(ascending=False).head(20))

print("\nDuplicate rows:", df.duplicated().sum())

Shape: (101766, 50)

Columns:
['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'payer_code', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted']

Missing values:
max_glu_serum               96420
A1Cresult                   84748
race                            0
gender                          0

In [51]:
drop_columns = [
    "weight",
    "payer_code",
    "medical_specialty",
    "encounter_id"
]

df = df.drop(
    columns=[col for col in drop_columns if col in df.columns],
    errors="ignore"
)

print("Remaining shape:", df.shape)

Remaining shape: (101766, 46)


In [52]:
if "patient_nbr" in df.columns:

    df = df.drop_duplicates(
        subset=["patient_nbr"]
    )

    df = df.drop(
        columns=["patient_nbr"]
    )

print("Shape after removing duplicate patients:", df.shape)

Shape after removing duplicate patients: (71518, 45)


In [53]:
# Create binary target
# <30 = Readmitted
# NO and >30 = Not Readmitted

df["target"] = (
    df["readmitted"] == "<30"
).astype(int)

print(df["target"].value_counts())

print(
    df["target"].value_counts(
        normalize=True
    ) * 100
)

target
0    65225
1     6293
Name: count, dtype: int64
target
0    91.200817
1     8.799183
Name: proportion, dtype: float64


In [54]:
id_columns = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id"
]

for col in id_columns:

    if col in df.columns:

        df[col] = df[col].astype(str)

print("ID columns converted to categorical strings")

ID columns converted to categorical strings


In [55]:
def convert_age(age):

    if pd.isna(age):
        return np.nan

    age = str(age)

    if "[" in age and "-" in age:

        values = age.strip("[]()").split("-")

        try:
            lower = int(values[0])
            upper = int(values[1])

            return (lower + upper) / 2

        except:
            return np.nan

    return np.nan


if "age" in df.columns:

    df["age"] = df["age"].apply(
        convert_age
    )

print(df["age"].head())

0     5.0
1    15.0
2    25.0
3    35.0
4    45.0
Name: age, dtype: float64


In [56]:
X = df.drop(
    columns=["readmitted", "target"],
    errors="ignore"
)

y = df["target"]

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

X shape: (71518, 44)
y shape: (71518,)

Target distribution:
target
0    65225
1     6293
Name: count, dtype: int64


In [57]:
feature_names = X.columns.tolist()

print("Number of features:", len(feature_names))

print(feature_names)

Number of features: 44
['race', 'gender', 'age', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed']


In [58]:
X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.2,

    random_state=42,

    stratify=y
)

print("Training samples:", len(X_train))

print("Testing samples:", len(X_test))

print("\nTraining distribution:")
print(y_train.value_counts())

print("\nTesting distribution:")
print(y_test.value_counts())

Training samples: 57214
Testing samples: 14304

Training distribution:
target
0    52180
1     5034
Name: count, dtype: int64

Testing distribution:
target
0    13045
1     1259
Name: count, dtype: int64


In [59]:
numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

Numerical features:
['age', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']

Categorical features:
['race', 'gender', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed']


C:\Users\vishnuvardhan\AppData\Local\Temp\ipykernel_43848\2895382912.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(


In [60]:
numeric_pipeline = Pipeline(
    steps=[

        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        )

    ]
)


categorical_pipeline = Pipeline(
    steps=[

        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),

        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )

    ]
)


preprocessor = ColumnTransformer(

    transformers=[

        (
            "num",
            numeric_pipeline,
            numeric_features
        ),

        (
            "cat",
            categorical_pipeline,
            categorical_features
        )

    ]
)

print("Preprocessor created successfully")

Preprocessor created successfully


In [61]:
X_train_processed = preprocessor.fit_transform(
    X_train
)

X_test_processed = preprocessor.transform(
    X_test
)

print(
    "Processed training shape:",
    X_train_processed.shape
)

print(
    "Processed testing shape:",
    X_test_processed.shape
)

Processed training shape: (57214, 2228)
Processed testing shape: (14304, 2228)


In [62]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train_processed
)

X_test_scaled = scaler.transform(
    X_test_processed
)

print("Scaling completed")

Scaling completed


In [46]:
!pip install catboost

^C


In [63]:
# ============================================================
# MODEL DEFINITIONS
# ============================================================

from catboost import CatBoostClassifier

scale_pos_weight = (
    (y_train == 0).sum()
    /
    (y_train == 1).sum()
)

models = {

    "Logistic Regression":
    LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ),

    "Decision Tree":
    DecisionTreeClassifier(
        class_weight="balanced",
        random_state=42
    ),

    "Random Forest":
    RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    "Extra Trees":
    ExtraTreesClassifier(
        n_estimators=100,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    "XGBoost":
    XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        scale_pos_weight=scale_pos_weight,
        eval_metric="logloss",
        random_state=42
    ),

    "CatBoost":
    CatBoostClassifier(
        iterations=200,
        depth=6,
        learning_rate=0.1,
        loss_function="Logloss",
        eval_metric="AUC",
        auto_class_weights="Balanced",
        random_seed=42,
        verbose=0
    ),

    "LightGBM":
    LGBMClassifier(
        n_estimators=100,
        learning_rate=0.1,
        class_weight="balanced",
        random_state=42,
        verbose=-1
    )
}

print("Models ready:")
for name in models:
    print("-", name)

Models ready:
- Logistic Regression
- Decision Tree
- Random Forest
- Extra Trees
- XGBoost
- CatBoost
- LightGBM


In [64]:
# ============================================================
# CELL 17: TRAIN ALL MODELS
# ============================================================

trained_models = {}

for name, model in models.items():

    print(f"Training {name}...")

    model.fit(
        X_train_scaled,
        y_train
    )

    trained_models[name] = model

print("\nAll models trained successfully!")
print("\nModels trained:")
for name in trained_models.keys():
    print("-", name)

Training Logistic Regression...
Training Decision Tree...
Training Random Forest...
Training Extra Trees...
Training XGBoost...
Training CatBoost...
Training LightGBM...
  Using cached catboost-1.2.10-cp314-cp314-win_amd64.whl.metadata (1.5 kB)
  Using cached graphviz-0.21-py3-none-any.whl.metadata (12 kB)
  Using cached plotly-6.9.0-py3-none-any.whl.metadata (9.0 kB)
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? et

ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'D:\\hospital_dashboard\\backend\\venv\\Lib\\site-packages\\catboost\\_catboost.pyd'
Check the permissions.


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached catboost-1.2.10-cp314-cp314-win_amd64.whl.metadata (1.5 kB)
  Using cached graphviz-0.21-py3-none-any.whl.metadata (12 kB)
  Using cached plotly-6.9.0-py3-none-any.whl.metadata (9.0 kB)
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.5/101.7 MB 221.9 kB/s eta 0:07:37
   ---------------------------------------- 0.5/101.7 MB 221.9 kB/s eta 0:07:37


ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'D:\\hospital_dashboard\\backend\\venv\\Lib\\site-packages\\catboost\\_catboost.pyd'
Check the permissions.


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



All models trained successfully!

Models trained:
- Logistic Regression
- Decision Tree
- Random Forest
- Extra Trees
- XGBoost
- CatBoost
- LightGBM


In [ ]:
# ============================================================
# CELL 18: NORMAL MODEL EVALUATION
# ============================================================

results = []

for name, model in trained_models.items():

    # Predict class
    y_pred = model.predict(
        X_test_scaled
    )

    # Predict probability
    y_prob = model.predict_proba(
        X_test_scaled
    )[:, 1]

    # Calculate metrics
    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )

    roc_auc = roc_auc_score(
        y_test,
        y_prob
    )

    results.append({

        "Model": name,

        "Accuracy": accuracy,

        "Precision": precision,

        "Recall": recall,

        "F1-Score": f1,

        "ROC-AUC": roc_auc

    })


# Create DataFrame
results_df = pd.DataFrame(
    results
)

print("Model evaluation completed!")

results_df